# Liora Engine · primera generación real (GPU gratuita, 0 €)

Ejecutar en **Kaggle Notebooks** (gratis, sin tarjeta; requiere verificar el teléfono para activar GPU e Internet):
*Settings → Accelerator: GPU T4 x2 o P100 · Internet: On*. Kaggle ofrece ~29 GB de RAM, necesarios para cargar Wan 2.2 5B.
Colab gratuito (12,7 GB de RAM) probablemente se queda corto en RAM: no verificado.

Este cuaderno **no simula nada**: si falta GPU, pesos o memoria, el motor se detiene y lo deja escrito en `manifest.json`.
La foto oficial de Liora se sube como *Dataset privado* de Kaggle (nunca al repositorio público).

In [ ]:
!nvidia-smi
!free -g && df -h /kaggle/working 2>/dev/null || df -h .

In [ ]:
# Código del motor (solo la carpeta liora/engine del repositorio)
!git clone --depth 1 --filter=blob:none --sparse https://github.com/alvarogavilan/edge-funding.git
%cd edge-funding
!git sparse-checkout set liora/engine
%cd liora/engine
# torch ya viene instalado en Kaggle/Colab: solo se añaden las dependencias del motor
!pip install -q "diffusers>=0.35" "transformers>=4.49" "accelerate>=1.1" ftfy sentencepiece imageio-ffmpeg dlib-bin face_recognition_models==0.3.0 mediapipe
!apt-get -qq install -y libegl1 libgles2 > /dev/null 2>&1 || true

In [ ]:
# 1) Medición real de la capacidad gratuita (se guarda en capacity.json)
!python -m liora_engine probe --bench --out /kaggle/working/capacity.json | head -60

In [ ]:
# 2) Foto oficial (Dataset privado de Kaggle). Ajusta la ruta.
import glob
cands = glob.glob('/kaggle/input/*/*.jp*g') + glob.glob('/kaggle/input/*/*.png') + glob.glob('/kaggle/input/*/*/*.jp*g') + glob.glob('/kaggle/input/*/*/*.png')
print(cands)
OFFICIAL = cands[0] if cands else None
assert OFFICIAL, 'Añade la foto oficial como Dataset privado (Add data → Upload)'

In [ ]:
# 3) Paquete de identidad: solo descriptores + SHA-256, sin imágenes
!python -m liora_engine identity-enroll --pack /kaggle/working/identity-pack.json --image "$OFFICIAL" --official

In [ ]:
# 4) Descarga de pesos de Wan 2.2 TI2V-5B (Apache-2.0) con medición de tiempo y tamaño
import time, os, subprocess
from huggingface_hub import snapshot_download
t0 = time.time()
path = snapshot_download('Wan-AI/Wan2.2-TI2V-5B-Diffusers')
size = int(subprocess.check_output(['du', '-sb', path]).split()[0])
print(path, round(size / 1e9, 1), 'GB en', round(time.time() - t0), 's')
os.environ['LIORA_WAN22_TI2V'] = path

In [ ]:
# 5) Trabajo de 1 escena de 5 s (121 fotogramas, 704×1280, 24 fps) partiendo de la foto oficial
!python -m liora_engine plan --official "$OFFICIAL" --pack /kaggle/working/identity-pack.json --out /kaggle/working/job.json --scenes 2 --seconds 5 --steps 40
!python -m liora_engine models

In [ ]:
# 6) Generación real de la etapa 1 con control de identidad, continuidad y fotogramas
!LIORA_WAN22_TI2V="$path" python -m liora_engine render --job /kaggle/working/job.json --work /kaggle/working/render --only 1
!cat /kaggle/working/render/manifest.json | head -120

In [ ]:
# 7) Ver el resultado aquí y descargarlo (Output → render/scene-01-*/attempt-*.mp4)
import glob
from IPython.display import Video
clips = sorted(glob.glob('/kaggle/working/render/scene-*/attempt-*[0-9].mp4'))
print(clips)
Video(clips[-1], embed=True, width=360) if clips else print('Sin clip: revisa manifest.json')

Si la etapa 1 se acepta, cambia `--only 1` por `--only 2` para generar la etapa 2 partiendo de su último fotograma.
Sube los MP4 a Liora Studio (Producción → subir clip) para la revisión humana obligatoria, y conserva `capacity.json` y
`manifest.json` como prueba reproducible (semilla, versiones, tiempos, huellas SHA-256).